In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. LOAD DATA
# ============================================================

calendar_path = "C:\\Users\\saksh\\OneDrive\\Desktop\\Resource\\Code\\geekforgeeks\\data\\raw\\calendar.csv.gz"
listings_path = "C:\\Users\\saksh\\OneDrive\\Desktop\\Resource\\Code\\geekforgeeks\\data\\raw\\rental-price-predict\\airbnb_test.csv"

calendar = pd.read_csv(calendar_path)
listings = pd.read_csv(listings_path)

print("Calendar shape:", calendar.shape)
print("Listings shape:", listings.shape)


# ============================================================
# 2. BASIC INFORMATION
# ============================================================

print("\n--- Calendar Columns ---")
print(calendar.columns.tolist())

print("\n--- Data Types ---")
print(calendar.dtypes)

print("\n--- First 5 Rows ---")
display(calendar.head())


# ============================================================
# 3. UNIQUE LISTINGS & DATES
# ============================================================

print("\n--- Unique Listings ---")
print(calendar["listing_id"].nunique())

calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")

print("\n--- Date Range ---")
print("Minimum date:", calendar["date"].min())
print("Maximum date:", calendar["date"].max())

print("\n--- Unique Dates ---")
print(calendar["date"].nunique())


# ============================================================
# 4. MISSING VALUES
# ============================================================

print("\n--- Missing Values ---")
missing = calendar.isnull().sum()
missing_pct = (calendar.isnull().mean() * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_pct
})

display(missing_report)


# ============================================================
# 5. DUPLICATE LISTING + DATE RECORDS
# ============================================================

duplicates = calendar.duplicated(
    subset=["listing_id", "date"]
).sum()

print("\n--- Duplicate listing_id + date rows ---")
print(duplicates)


# ============================================================
# 6. AVAILABLE VS UNAVAILABLE
# ============================================================

print("\n--- Availability ---")

print(calendar["available"].value_counts(dropna=False))

availability_pct = (
    calendar["available"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("\nPercentage:")
print(availability_pct)


# ============================================================
# 7. CLEAN PRICE COLUMN
# ============================================================

calendar["price_numeric"] = (
    calendar["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .replace(["nan", "None", ""], np.nan)
    .astype(float)
)

print("\n--- Price Statistics ---")
display(calendar["price_numeric"].describe())

print("\nPrice missing values:")
print(calendar["price_numeric"].isna().sum())


# ============================================================
# 8. PRICE DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 5))

plt.hist(
    calendar["price_numeric"].dropna(),
    bins=100
)

plt.xlabel("Advertised Price ($)")
plt.ylabel("Number of Records")
plt.title("Inside Airbnb London — Price Distribution")

plt.show()


# ============================================================
# 9. OVERLAP WITH KAGGLE LISTINGS
# ============================================================

# Make sure IDs are comparable
calendar_ids = set(calendar["listing_id"].astype(str))
listing_ids = set(listings["id"].astype(str))

overlap = calendar_ids.intersection(listing_ids)

print("\n--- Dataset Overlap ---")
print("Calendar unique listings:", len(calendar_ids))
print("Kaggle unique listings:", len(listing_ids))
print("Overlapping listings:", len(overlap))

calendar_overlap_pct = (
    len(overlap) / len(calendar_ids) * 100
)

print(
    f"Percentage of calendar listings found in Kaggle: "
    f"{calendar_overlap_pct:.2f}%"
)


# ============================================================
# 10. RECORD COVERAGE
# ============================================================

overlap_records = calendar[
    calendar["listing_id"].astype(str).isin(overlap)
]

print("\n--- Overlapping Calendar Records ---")
print("Rows:", len(overlap_records))

print(
    "Percentage of calendar rows belonging to overlapping listings:",
    round(len(overlap_records) / len(calendar) * 100, 2),
    "%"
)


# ============================================================
# 11. RECORDS PER LISTING
# ============================================================

records_per_listing = calendar.groupby(
    "listing_id"
).size()

print("\n--- Records Per Listing ---")

display(records_per_listing.describe())

print(
    "Average records/listing:",
    round(records_per_listing.mean(), 2)
)


# ============================================================
# 12. DATE COVERAGE
# ============================================================

date_counts = calendar.groupby("date").size()

print("\n--- Records Per Date ---")
display(date_counts.describe())

print("\nFirst 10 dates:")
display(date_counts.head(10))


# ============================================================
# 13. QUICK SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("V2.0 CALENDAR DATA SUMMARY")
print("=" * 60)

print(f"Rows: {len(calendar):,}")
print(f"Columns: {calendar.shape[1]}")
print(f"Unique listings: {calendar['listing_id'].nunique():,}")
print(f"Unique dates: {calendar['date'].nunique():,}")
print(f"Date range: {calendar['date'].min().date()} → {calendar['date'].max().date()}")
print(f"Duplicate listing/date rows: {duplicates:,}")
print(f"Overlapping listings with Kaggle: {len(overlap):,}")
print(f"Overlap percentage: {calendar_overlap_pct:.2f}%")
print(f"Average records/listing: {records_per_listing.mean():.2f}")

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Load London Inside Airbnb data
listings = pd.read_csv(r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\listings.csv.gz")
calendar = pd.read_csv(r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\calendar.csv.gz")

print("Listings shape:", listings.shape)
print("Calendar shape:", calendar.shape)

# Convert IDs to string for safe comparison
listing_ids = set(listings["id"].astype(str))
calendar_ids = set(calendar["listing_id"].astype(str))

overlap = listing_ids & calendar_ids

print("\n--- ID MATCH ---")
print("Unique listings:", len(listing_ids))
print("Unique calendar listings:", len(calendar_ids))
print("Matching listings:", len(overlap))

print(
    "Listings covered by calendar:",
    round(len(overlap) / len(listing_ids) * 100, 2),
    "%"
)

# Check calendar coverage per listing
records_per_listing = calendar.groupby("listing_id").size()

print("\n--- CALENDAR COVERAGE ---")
print(records_per_listing.describe())

# Date information
calendar["date"] = pd.to_datetime(calendar["date"])

print("\nDate range:")
print(calendar["date"].min(), "→", calendar["date"].max())

print("\nUnique dates:", calendar["date"].nunique())

# Availability
print("\n--- AVAILABILITY ---")
print(calendar["available"].value_counts(normalize=True) * 100)

# Price
calendar["price_numeric"] = (
    calendar["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

print("\n--- PRICE ---")
print(calendar["price_numeric"].describe())

Listings shape: (92638, 90)
Calendar shape: (33871636, 5)

--- ID MATCH ---
Unique listings: 92638
Unique calendar listings: 92799
Matching listings: 92638
Listings covered by calendar: 100.0 %

--- CALENDAR COVERAGE ---
count    92799.000000
mean       365.000011
std          0.003283
min        365.000000
25%        365.000000
50%        365.000000
75%        365.000000
max        366.000000
dtype: float64

Date range:
2026-06-19 00:00:00 → 2027-06-30 00:00:00

Unique dates: 377

--- AVAILABILITY ---
available
f    58.092024
t    41.907976
Name: proportion, dtype: float64


KeyError: 'price'

In [ ]:
print(calendar.columns.tolist())
display(calendar.head())

['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']


,listing_id,date,available,minimum_nights,maximum_nights
0,11551,2026-06-25,f,1,1125
1,11551,2026-06-26,f,1,1125
2,11551,2026-06-27,f,1,1125
3,11551,2026-06-28,f,1,1125
4,11551,2026-06-29,f,1,1125


In [ ]:
import pandas as pd

# Load data

listings = pd.read_csv(r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\listings.csv.gz")
calendar = pd.read_csv(r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\calendar.csv.gz")

# Date
calendar["date"] = pd.to_datetime(calendar["date"])

# Convert availability to 0/1
calendar["is_available"] = (calendar["available"] == "t").astype(int)

# --------------------------------------------------
# 1. CREATE DATE FEATURES
# --------------------------------------------------

calendar["day_of_week"] = calendar["date"].dt.dayofweek
calendar["day_name"] = calendar["date"].dt.day_name()
calendar["month"] = calendar["date"].dt.month
calendar["month_name"] = calendar["date"].dt.month_name()
calendar["is_weekend"] = calendar["day_of_week"] >= 5

# --------------------------------------------------
# 2. DAILY MARKET AVAILABILITY
# --------------------------------------------------

daily_market = (
    calendar
    .groupby("date")
    .agg(
        total_listings=("listing_id", "nunique"),
        available_listings=("is_available", "sum")
    )
    .reset_index()
)

daily_market["unavailable_listings"] = (
    daily_market["total_listings"]
    - daily_market["available_listings"]
)

daily_market["availability_rate"] = (
    daily_market["available_listings"]
    / daily_market["total_listings"]
)

daily_market["unavailability_rate"] = (
    1 - daily_market["availability_rate"]
)

# --------------------------------------------------
# 3. TEMPORAL PATTERNS
# --------------------------------------------------

print("DAILY MARKET SUMMARY")
display(daily_market.head())

print("\nAvailability statistics:")
display(daily_market["availability_rate"].describe())

print("\nAverage availability by month:")
display(
    calendar.groupby("month")["is_available"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nAverage availability: weekday vs weekend")

display(
    calendar.groupby("is_weekend")["is_available"]
    .mean()
    .mul(100)
    .round(2)
)

print("\nAvailability by day:")
display(
    calendar.groupby("day_name")["is_available"]
    .mean()
    .mul(100)
    .round(2)
)

# --------------------------------------------------
# 4. MERGE MARKET SIGNAL BACK TO CALENDAR
# --------------------------------------------------

calendar_v2 = calendar.merge(
    daily_market[
        [
            "date",
            "total_listings",
            "available_listings",
            "unavailable_listings",
            "availability_rate",
            "unavailability_rate"
        ]
    ],
    on="date",
    how="left"
)

print("\nV2 Calendar shape:", calendar_v2.shape)

display(calendar_v2.head())

DAILY MARKET SUMMARY


,date,total_listings,available_listings,unavailable_listings,availability_rate,unavailability_rate
0,2026-06-19,2189,394,1795,0.179991,0.820009
1,2026-06-20,15955,3453,12502,0.216421,0.783579
2,2026-06-21,19475,8034,11441,0.412529,0.587471
3,2026-06-22,19475,8906,10569,0.457304,0.542696
4,2026-06-23,19475,8906,10569,0.457304,0.542696



Availability statistics:


count    377.000000
mean       0.415986
std        0.065532
min        0.166415
25%        0.363916
50%        0.443873
75%        0.462602
max        0.502150
Name: availability_rate, dtype: float64


Average availability by month:


...

is_weekend
False    42.04
True     41.57
Name: is_available, dtype: float64


Availability by day:


day_name
Friday       41.27
Monday       42.35
Saturday     40.98
Sunday       42.16
Thursday     41.99
Tuesday      42.36
Wednesday    42.25
Name: is_available, dtype: float64


V2 Calendar shape: (33871636, 16)


,listing_id,date,available,minimum_nights,maximum_nights,is_available,day_of_week,day_name,month,month_name,is_weekend,total_listings,available_listings,unavailable_listings,availability_rate,unavailability_rate
0,11551,2026-06-25,f,1,1125,0,3,Thursday,6,June,False,33911,10909,23002,0.321695,0.678305
1,11551,2026-06-26,f,1,1125,0,4,Friday,6,June,False,50260,11202,39058,0.222881,0.777119
2,11551,2026-06-27,f,1,1125,0,5,Saturday,6,June,True,64862,10794,54068,0.166415,0.833585
3,11551,2026-06-28,f,1,1125,0,6,Sunday,6,June,True,68796,17320,51476,0.251759,0.748241
4,11551,2026-06-29,f,1,1125,0,0,Monday,6,June,False,92299,21928,70371,0.237576,0.762424


In [ ]:
# ============================================================
# V2.0 STEP 4 — MARKET PRESSURE SCORE
# ============================================================

# Market pressure is the inverse of availability.
# Higher pressure = fewer listings available.

daily_market["market_pressure"] = (
    1 - daily_market["availability_rate"]
)

# Convert to a 0–100 score using min-max normalization
min_pressure = daily_market["market_pressure"].min()
max_pressure = daily_market["market_pressure"].max()

daily_market["market_pressure_score"] = (
    (daily_market["market_pressure"] - min_pressure)
    / (max_pressure - min_pressure)
) * 100

# Round for readability
daily_market["market_pressure_score"] = (
    daily_market["market_pressure_score"].round(2)
)

# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

print("Market Pressure Statistics:")
display(
    daily_market["market_pressure_score"].describe()
)

print("\nHighest pressure dates:")
display(
    daily_market[
        [
            "date",
            "availability_rate",
            "unavailability_rate",
            "market_pressure_score"
        ]
    ]
    .sort_values("market_pressure_score", ascending=False)
    .head(10)
)

print("\nLowest pressure dates:")
display(
    daily_market[
        [
            "date",
            "availability_rate",
            "unavailability_rate",
            "market_pressure_score"
        ]
    ]
    .sort_values("market_pressure_score")
    .head(10)
)

Market Pressure Statistics:


count    377.000000
mean      25.664111
std       19.519106
min        0.000000
25%       11.780000
50%       17.360000
75%       41.170000
max      100.000000
Name: market_pressure_score, dtype: float64


Highest pressure dates:


,date,availability_rate,unavailability_rate,market_pressure_score
8,2026-06-27,0.166415,0.833585,100.00
15,2026-07-04,0.173256,0.826744,97.96
0,2026-06-19,0.179991,0.820009,95.96
14,2026-07-03,0.182437,0.817563,95.23
1,2026-06-20,0.216421,0.783579,85.11
7,2026-06-26,0.222881,0.777119,83.18
372,2027-06-26,0.232634,0.767366,80.28
13,2026-07-02,0.235875,0.764125,79.31
10,2026-06-29,0.237576,0.762424,78.80
16,2026-07-05,0.244302,0.755698,76.80



Lowest pressure dates:


,date,availability_rate,unavailability_rate,market_pressure_score
75,2026-09-02,0.502150,0.497850,0.00
74,2026-09-01,0.500684,0.499316,0.44
73,2026-08-31,0.498691,0.501309,1.03
67,2026-08-25,0.496967,0.503033,1.54
88,2026-09-15,0.496525,0.503475,1.68
87,2026-09-14,0.495943,0.504057,1.85
68,2026-08-26,0.494035,0.505965,2.42
86,2026-09-13,0.493303,0.506697,2.64
80,2026-09-07,0.493281,0.506719,2.64
89,2026-09-16,0.492958,0.507042,2.74


In [ ]:
# ============================================================
# V2.0 STEP 5 — ATTACH MARKET PRESSURE TO CALENDAR
# ============================================================

market_features = daily_market[
    [
        "date",
        "availability_rate",
        "market_pressure_score"
    ]
]

calendar_v2 = calendar.merge(
    market_features,
    on="date",
    how="left"
)

print("V2 dataset shape:", calendar_v2.shape)

print("\nMissing market features:")
print(calendar_v2[
    ["availability_rate", "market_pressure_score"]
].isna().sum())

print("\nSample:")
display(
    calendar_v2[
        [
            "listing_id",
            "date",
            "available",
            "minimum_nights",
            "maximum_nights",
            "availability_rate",
            "market_pressure_score"
        ]
    ].head(10)
)

MemoryError: Unable to allocate 775. MiB for an array with shape (3, 33871636) and data type object

In [ ]:
# ============================================================
# V2.0 STEP 5 — CREATE A LIGHTWEIGHT DATE FEATURE TABLE
# ============================================================

date_features = daily_market[
    [
        "date",
        "availability_rate",
        "market_pressure_score"
    ]
].copy()

# Add temporal features
date_features["day_of_week"] = date_features["date"].dt.dayofweek
date_features["is_weekend"] = date_features["day_of_week"] >= 5
date_features["month"] = date_features["date"].dt.month
date_features["month_name"] = date_features["date"].dt.month_name()

print("Date feature table:", date_features.shape)

display(date_features.head())

print("\nMemory usage:")
print(
    date_features.memory_usage(deep=True).sum() / 1024**2,
    "MB"
)

Date feature table: (377, 7)


,date,availability_rate,market_pressure_score,day_of_week,is_weekend,month,month_name
0,2026-06-19,0.179991,95.96,4,False,6,June
1,2026-06-20,0.216421,85.11,5,True,6,June
2,2026-06-21,0.412529,26.69,6,True,6,June
3,2026-06-22,0.457304,13.36,0,False,6,June
4,2026-06-23,0.457304,13.36,1,False,6,June



Memory usage:
0.03179359436035156 MB


In [ ]:
import pandas as pd
listings = pd.read_csv(
    r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\listings.csv.gz"
)

calendar = pd.read_csv(
    r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\calendar.csv.gz"
)

In [ ]:
print(listings.columns.tolist())

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'minimum_nig

In [ ]:
# ============================================================
# V2.0 STEP 6 — PREPARE PROPERTY FEATURES
# ============================================================

property_features = listings[
    [
        "id",
        "latitude",
        "longitude",
        "neighbourhood_cleansed",
        "property_type",
        "room_type",
        "accommodates",
        "bathrooms",
        "bedrooms",
        "beds",
        "host_is_superhost",
        "host_listings_count",
        "host_response_rate",
        "host_acceptance_rate",
        "number_of_reviews",
        "number_of_reviews_ltm",
        "number_of_reviews_l30d",
        "review_scores_rating",
        "review_scores_accuracy",
        "review_scores_cleanliness",
        "review_scores_checkin",
        "review_scores_communication",
        "review_scores_location",
        "review_scores_value",
        "reviews_per_month",
        "amenities",
        "price"
    ]
].copy()

print("Property feature table shape:", property_features.shape)

display(property_features.head())

print("\nMissing values:")
display(
    property_features.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

Property feature table shape: (92638, 27)


,id,latitude,longitude,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bedrooms,beds,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,reviews_per_month,amenities,price
0,11551,51.46095,-0.11758,Lambeth,Entire rental unit,Entire home/apt,5,1.0,1.0,3.0,...,4.55,4.60,4.57,4.77,4.84,4.54,4.49,0.99,"[""Carbon monoxide alarm"", ""Portable air condit...",$234.50
1,13913,51.56861,-0.11270,Islington,Private room in rental unit,Private room,1,1.0,NaN,1.0,...,4.86,4.80,4.80,4.82,4.88,4.79,4.79,0.30,"[""Babysitter recommendations"", ""Coffee maker: ...",$127.00
2,15400,51.48780,-0.16813,Kensington and Chelsea,Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,...,4.81,4.86,4.87,4.88,4.84,4.94,4.74,0.49,"[""Refrigerator"", ""Long term stays allowed"", ""C...",$137.50
3,17402,51.52195,-0.14094,Westminster,Entire rental unit,Entire home/apt,6,2.0,3.0,3.0,...,4.77,4.84,4.73,4.73,4.73,4.89,4.62,0.31,"[""Drying rack for clothing"", ""Dishwasher"", ""Lo...",$606.67
4,513198,51.48875,-0.10934,Lambeth,Private room in rental unit,Private room,1,1.5,1.0,1.0,...,4.42,4.67,3.63,4.79,4.74,4.72,4.42,0.25,"[""Oven"", ""City skyline view"", ""Toaster"", ""Priv...",$68.30



Missing values:


host_acceptance_rate           92638
host_response_rate             92638
bathrooms                      34876
beds                           32475
price                          30398
bedrooms                       23983
review_scores_value            21966
review_scores_location         21966
review_scores_checkin          21965
review_scores_communication    21946
review_scores_accuracy         21941
review_scores_cleanliness      21936
reviews_per_month              21927
review_scores_rating           21927
host_is_superhost                 76
dtype: int64

In [ ]:
# ============================================================
# V2.0 STEP 7 — INSPECT PRICE TARGET
# ============================================================

price = (
    property_features["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

property_features["price_numeric"] = pd.to_numeric(
    price,
    errors="coerce"
)

print("Price statistics:")
display(property_features["price_numeric"].describe())

print("\nMissing prices:")
print(property_features["price_numeric"].isna().sum())

print("\nZero / negative prices:")
print(
    (property_features["price_numeric"] <= 0).sum()
)

print("\nPrice percentiles:")
print(
    property_features["price_numeric"]
    .quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
)

Price statistics:


count     62240.000000
mean        271.469220
std        2190.599006
min           2.230000
25%         100.000000
50%         180.000000
75%         300.032500
max      527524.000000
Name: price_numeric, dtype: float64


Missing prices:
30398

Zero / negative prices:
0

Price percentiles:
0.01      35.0000
0.05      50.7490
0.25     100.0000
0.50     180.0000
0.75     300.0325
0.95     676.0000
0.99    1373.4575
Name: price_numeric, dtype: float64


In [ ]:
# ============================================================
# V2.0 STEP 8 — CLEAN BASE PRICE
# ============================================================

# Keep listings with a valid price
priced = property_features[
    property_features["price_numeric"].notna()
].copy()

# Remove extreme outliers using the 99th percentile
upper_limit = priced["price_numeric"].quantile(0.99)

priced_clean = priced[
    priced["price_numeric"] <= upper_limit
].copy()

print("Original priced listings:", len(priced))
print("99th percentile:", round(upper_limit, 2))
print("Clean priced listings:", len(priced_clean))

print("\nClean price statistics:")
display(priced_clean["price_numeric"].describe())

Original priced listings: 62240
99th percentile: 1373.46
Clean priced listings: 61617

Clean price statistics:


count    61617.000000
mean       232.944402
std        195.311137
min          2.230000
25%         99.000000
50%        179.000000
75%        296.000000
max       1373.000000
Name: price_numeric, dtype: float64

In [ ]:
# ============================================================
# V2.0 STEP 9 — ANALYZE PRICE VS MARKET PRESSURE
# ============================================================

# Add market pressure to each listing through a small date sample.
# First create a random sample of calendar records.

calendar_sample = calendar.sample(
    n=500_000,
    random_state=42
).copy()

# Attach daily market pressure
calendar_sample = calendar_sample.merge(
    date_features[
        ["date", "market_pressure_score"]
    ],
    on="date",
    how="left"
)

# Create pressure buckets
calendar_sample["pressure_level"] = pd.cut(
    calendar_sample["market_pressure_score"],
    bins=[-1, 20, 40, 60, 80, 101],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

print("Pressure distribution:")
display(
    calendar_sample["pressure_level"]
    .value_counts()
    .sort_index()
)

NameError: name 'date_features' is not defined

In [ ]:
# Recreate date_features if needed

date_features = daily_market[
    [
        "date",
        "availability_rate",
        "market_pressure_score"
    ]
].copy()

date_features["day_of_week"] = date_features["date"].dt.dayofweek
date_features["is_weekend"] = date_features["day_of_week"] >= 5
date_features["month"] = date_features["date"].dt.month
date_features["month_name"] = date_features["date"].dt.month_name()

print(date_features.shape)
display(date_features.head())

NameError: name 'daily_market' is not defined

In [14]:
import pandas as pd

# ============================================================
# REBUILD V2.0 DATE-LEVEL MARKET FEATURES
# ============================================================

calendar = pd.read_csv(
    r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\calendar.csv.gz"
)

# Convert date
calendar["date"] = pd.to_datetime(calendar["date"])

# Availability: t = available, f = unavailable
calendar["is_available"] = (
    calendar["available"] == "t"
).astype("int8")

# ------------------------------------------------------------
# DAILY MARKET STATISTICS
# ------------------------------------------------------------

daily_market = (
    calendar
    .groupby("date")
    .agg(
        total_listings=("listing_id", "nunique"),
        available_listings=("is_available", "sum")
    )
    .reset_index()
)

daily_market["unavailable_listings"] = (
    daily_market["total_listings"]
    - daily_market["available_listings"]
)

daily_market["availability_rate"] = (
    daily_market["available_listings"]
    / daily_market["total_listings"]
)

daily_market["unavailability_rate"] = (
    1 - daily_market["availability_rate"]
)

# ------------------------------------------------------------
# MARKET PRESSURE
# ------------------------------------------------------------

daily_market["market_pressure"] = (
    1 - daily_market["availability_rate"]
)

min_pressure = daily_market["market_pressure"].min()
max_pressure = daily_market["market_pressure"].max()

daily_market["market_pressure_score"] = (
    (daily_market["market_pressure"] - min_pressure)
    / (max_pressure - min_pressure)
) * 100

daily_market["market_pressure_score"] = (
    daily_market["market_pressure_score"].round(2)
)

# ------------------------------------------------------------
# TEMPORAL FEATURES
# ------------------------------------------------------------

date_features = daily_market[
    [
        "date",
        "availability_rate",
        "unavailability_rate",
        "market_pressure_score"
    ]
].copy()

date_features["day_of_week"] = date_features["date"].dt.dayofweek
date_features["day_name"] = date_features["date"].dt.day_name()
date_features["is_weekend"] = date_features["day_of_week"] >= 5
date_features["month"] = date_features["date"].dt.month
date_features["month_name"] = date_features["date"].dt.month_name()

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print("daily_market:", daily_market.shape)
print("date_features:", date_features.shape)

display(date_features.head())

daily_market: (377, 8)
date_features: (377, 9)


,date,availability_rate,unavailability_rate,market_pressure_score,day_of_week,day_name,is_weekend,month,month_name
0,2026-06-19,0.179991,0.820009,95.96,4,Friday,False,6,June
1,2026-06-20,0.216421,0.783579,85.11,5,Saturday,True,6,June
2,2026-06-21,0.412529,0.587471,26.69,6,Sunday,True,6,June
3,2026-06-22,0.457304,0.542696,13.36,0,Monday,False,6,June
4,2026-06-23,0.457304,0.542696,13.36,1,Tuesday,False,6,June


In [15]:
# ============================================================
# V2.0 STEP 6 — LONDON PROPERTY PRICE BASELINE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ------------------------------------------------------------
# 1. PREPARE DATA
# ------------------------------------------------------------

df = property_features.copy()

# Remove missing prices
df = df[df["price_numeric"].notna()].copy()

# Remove extreme top 1%
upper_limit = df["price_numeric"].quantile(0.99)

df = df[
    df["price_numeric"] <= upper_limit
].copy()

print("Training rows:", len(df))
print("Price upper limit:", round(upper_limit, 2))


# ------------------------------------------------------------
# 2. FEATURES
# ------------------------------------------------------------

numeric_features = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "host_listings_count",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "review_scores_rating",
    "review_scores_location",
    "reviews_per_month"
]

categorical_features = [
    "neighbourhood_cleansed",
    "property_type",
    "room_type",
    "host_is_superhost"
]

X = df[numeric_features + categorical_features]
y = df["price_numeric"]


# ------------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ------------------------------------------------------------
# 4. PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# ------------------------------------------------------------
# 5. MODEL
# ------------------------------------------------------------

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])


# ------------------------------------------------------------
# 6. TRAIN
# ------------------------------------------------------------

print("\nTraining Random Forest...")

pipeline.fit(X_train, y_train)


# ------------------------------------------------------------
# 7. EVALUATE
# ------------------------------------------------------------

predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("\n==============================")
print("LONDON BASELINE RESULTS")
print("==============================")

print(f"R²   : {r2:.4f}")
print(f"MAE  : ${mae:.2f}")
print(f"RMSE : ${rmse:.2f}")

Training rows: 61617
Price upper limit: 1373.46

Training Random Forest...

LONDON BASELINE RESULTS
R²   : 0.6708
MAE  : $66.86
RMSE : $112.64


In [ ]:
import pandas as pd
import numpy as np

# Make sure date is datetime
calendar["date"] = pd.to_datetime(calendar["date"])

# -----------------------------
# 1. Aggregate market by date
# -----------------------------
daily_market = (
    calendar.groupby("date")
    .agg(
        total_listings=("listing_id", "nunique"),
        available_listings=("available", "sum")
    )
    .reset_index()
)

daily_market["unavailable_listings"] = (
    daily_market["total_listings"] -
    daily_market["available_listings"]
)

# -----------------------------
# 2. Market pressure
# -----------------------------
daily_market["availability_rate"] = (
    daily_market["available_listings"] /
    daily_market["total_listings"]
)

daily_market["unavailability_rate"] = (
    daily_market["unavailable_listings"] /
    daily_market["total_listings"]
)

# Higher = more listings unavailable
daily_market["market_pressure"] = daily_market["unavailability_rate"]

# Normalize to 0–100
min_pressure = daily_market["market_pressure"].min()
max_pressure = daily_market["market_pressure"].max()

daily_market["market_pressure_score"] = (
    (daily_market["market_pressure"] - min_pressure) /
    (max_pressure - min_pressure)
) * 100

# -----------------------------
# 3. Date features
# -----------------------------
date_features = daily_market.copy()

date_features["day_of_week"] = date_features["date"].dt.dayofweek

date_features["day_name"] = (
    date_features["date"].dt.day_name()
)

date_features["is_weekend"] = (
    date_features["day_of_week"] >= 5
).astype(int)

date_features["month"] = (
    date_features["date"].dt.month
)

date_features["month_name"] = (
    date_features["date"].dt.month_name()
)

# -----------------------------
# 4. Save
# -----------------------------
date_features.to_csv(
    "data/processed/london_date_features.csv",
    index=False
)

print("Saved:", len(date_features), "dates")
print(date_features.head())

In [2]:
import pandas as pd

calendar = pd.read_csv(
    "C:\\Users\\saksh\\OneDrive\\Desktop\\Resource\\Code\\geekforgeeks\\data\\raw\\calendar.csv.gz",
    compression="gzip"
)

print(calendar.shape)
print(calendar.head())

(33871636, 5)
   listing_id        date available  minimum_nights  maximum_nights
0       11551  2026-06-25         f               1            1125
1       11551  2026-06-26         f               1            1125
2       11551  2026-06-27         f               1            1125
3       11551  2026-06-28         f               1            1125
4       11551  2026-06-29         f               1            1125


In [3]:
calendar["date"] = pd.to_datetime(calendar["date"])

daily_market = (
    calendar.groupby("date")
    .agg(
        total_listings=("listing_id", "size"),
        available_listings=("available", "sum")
    )
    .reset_index()
)

daily_market["unavailable_listings"] = (
    daily_market["total_listings"]
    - daily_market["available_listings"]
)

daily_market["availability_rate"] = (
    daily_market["available_listings"]
    / daily_market["total_listings"]
)

daily_market["unavailability_rate"] = (
    daily_market["unavailable_listings"]
    / daily_market["total_listings"]
)

daily_market["market_pressure"] = (
    daily_market["unavailability_rate"]
)

daily_market["market_pressure_score"] = (
    (daily_market["market_pressure"]
     - daily_market["market_pressure"].min())
    /
    (daily_market["market_pressure"].max()
     - daily_market["market_pressure"].min())
) * 100

daily_market["day_of_week"] = daily_market["date"].dt.dayofweek
daily_market["day_name"] = daily_market["date"].dt.day_name()
daily_market["is_weekend"] = (
    daily_market["day_of_week"] >= 5
).astype(int)
daily_market["month"] = daily_market["date"].dt.month
daily_market["month_name"] = daily_market["date"].dt.month_name()

date_features = daily_market.copy()

date_features.to_csv(
    "data/processed/london_date_features.csv",
    index=False
)

print("DONE:", date_features.shape)

TypeError: unsupported operand type(s) for -: 'int' and 'str'

In [4]:
date_features = pd.read_csv(
    "data/processed/london_date_features.csv"
)

date_features["date"] = pd.to_datetime(date_features["date"])

print(date_features.shape)
print(date_features.head())

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/london_date_features.csv'

In [5]:
import pandas as pd
import os

calendar_path = r"C:\Users\saksh\OneDrive\Desktop\Resource\Code\geekforgeeks\data\raw\calendar.csv.gz"

# Store daily totals here
daily_parts = []

print("Starting chunked processing...")

for chunk in pd.read_csv(
    calendar_path,
    compression="gzip",
    usecols=["listing_id", "date", "available"],
    chunksize=500_000
):
    # Convert availability to 0/1
    chunk["available"] = (
        chunk["available"]
        .map({"t": 1, "f": 0})
    )

    # Aggregate this chunk
    part = (
        chunk.groupby("date")
        .agg(
            total_listings=("listing_id", "size"),
            available_listings=("available", "sum")
        )
        .reset_index()
    )

    daily_parts.append(part)

    print(".", end="", flush=True)

print("\nCombining chunks...")

# Combine all chunk-level results
daily_market = (
    pd.concat(daily_parts)
    .groupby("date")
    .agg(
        total_listings=("total_listings", "sum"),
        available_listings=("available_listings", "sum")
    )
    .reset_index()
)

# -------------------------
# Calculate market metrics
# -------------------------

daily_market["unavailable_listings"] = (
    daily_market["total_listings"]
    - daily_market["available_listings"]
)

daily_market["availability_rate"] = (
    daily_market["available_listings"]
    / daily_market["total_listings"]
)

daily_market["unavailability_rate"] = (
    daily_market["unavailable_listings"]
    / daily_market["total_listings"]
)

daily_market["market_pressure"] = (
    daily_market["unavailability_rate"]
)

# Normalize 0–100
min_p = daily_market["market_pressure"].min()
max_p = daily_market["market_pressure"].max()

daily_market["market_pressure_score"] = (
    (daily_market["market_pressure"] - min_p)
    / (max_p - min_p)
) * 100

# -------------------------
# Date features
# -------------------------

daily_market["date"] = pd.to_datetime(daily_market["date"])

daily_market["day_of_week"] = daily_market["date"].dt.dayofweek
daily_market["day_name"] = daily_market["date"].dt.day_name()
daily_market["is_weekend"] = (
    daily_market["day_of_week"] >= 5
).astype(int)

daily_market["month"] = daily_market["date"].dt.month
daily_market["month_name"] = daily_market["date"].dt.month_name()

# -------------------------
# Save
# -------------------------

os.makedirs("data/processed", exist_ok=True)

date_features = daily_market.copy()

date_features.to_csv(
    "data/processed/london_date_features.csv",
    index=False
)

print("\n==============================")
print("DONE")
print("==============================")
print("Rows:", len(date_features))
print("Columns:", len(date_features.columns))
print("Saved to:")
print("data/processed/london_date_features.csv")

print("\nDate range:")
print(date_features["date"].min(), "→", date_features["date"].max())

print("\nPreview:")
print(date_features.head())

Starting chunked processing...
....................................................................
Combining chunks...

DONE
Rows: 377
Columns: 13
Saved to:
data/processed/london_date_features.csv

Date range:
2026-06-19 00:00:00 → 2027-06-30 00:00:00

Preview:
        date  total_listings  available_listings  unavailable_listings  \
0 2026-06-19            2189                 394                  1795   
1 2026-06-20           15955                3453                 12502   
2 2026-06-21           19475                8034                 11441   
3 2026-06-22           19475                8906                 10569   
4 2026-06-23           19475                8906                 10569   

   availability_rate  unavailability_rate  market_pressure  \
0           0.179991             0.820009         0.820009   
1           0.216421             0.783579         0.783579   
2           0.412529             0.587471         0.587471   
3           0.457304             0.542696   